# Multi-agent SAC for MPE and more 

* MASAC: Multi-Agent Soft Actor-Critic for Mixed Cooperative-Competitive Environments (no papers found)
* MPE: Multi-Agent Particle Environment using pettingzoo library [github](https://github.com/Farama-Foundation/PettingZoo)
* SAC: Soft Actor-Critic (2018) [arxiv](https://arxiv.org/abs/1812.05905)

## MASAC Algorithm 

**MASAC**(Multi-Agent Soft Actor-Critic) extends the maximum entropy reinforcement learning approach of SAC to multi-agent environments, combining SAC's stochastic policies and entropy regularization with MADDPG's centralized critic and decentralized actor training framework.

#### Key Components 

MASAC incorporates the following distinctive features:

* **Twin Centralized Critics**: Each agent maintains two soft centralized Q-networks that incorporate joint observations and actions, plus an entropy bonus to reduce overestimation bias (similar to MATD3).
* **Stochastic Policies**: Unlike MADDPG/MATD3's deterministic policies, MASAC uses stochastic policies that output a Gaussian distribution over actions (with a mean and standard deviation).
* **Entropy Regularization**: MASAC adds an entropy term to the policy loss to encourage exploration. 
* **Temperature Parameter**: An adjustable entropy weight $\alpha$ balances exploration and exploitation.
* **Automatic Entropy Tuning**: Optional (often beneficial) mechanism to automatically adjust the temperature parameter $\alpha$ to achieve a target entropy level.

#### Twin Critics Update 

For each agent i, maintain two soft Q-functions $Q_{\theta_{i,1}}$ and $Q_{\theta_{i,2}}$:

$$ 
\begin{aligned} 
\mathcal{L}(\theta_{i,k}) &= \mathbb{E}_{\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}'}[ (Q_{\theta_{i,k}} (\vec{o}, a_1, \dots, a_N) - y_i)^2 ] & \text{for } k \in {1,2} \\

\text{where } y_i &= r_i + \gamma \mathbb{E}_{a'_j \sim \pi_j(o'_j)} \left[ \min_{k=1,2} Q_{\overline{\theta}_{i,k}} (\vec{o}', a'_1, \dots, a'_N) - \alpha_i \log \pi_i(a'_i|o'_i) \right] 
\end{aligned} 
$$

,where $\vec{\pi}$ are the policies and $Q_{\overline{\theta}_{i,k}}$ target Q-functions with softly-updated parameters and $Q_{\theta_{i,k}}$
online Q-functions with current parameters.

#### Policy Update 

Unline MADDPG/MATD3 deterministic policy gradients, MASAC uses the reparameterization trick to sample actions from a Gaussian policy:

$$ 
\nabla_{\theta_i} J({\pi_i}) = \mathbb{E}_{\vec{o} \sim \mathcal{D},\epsilon \sim \mathcal{N}} \left[ \nabla_{\theta_i} \left( \alpha_i \log \pi_i(a_i | o_i) - \min_{k=1,2} Q_{\theta_{i,k}}(\vec{o}, a_1, \dots,a_i, \dots, a_N) \right) \right] \ \text{where } a_i = f_{\theta_i}(\epsilon_i; o_i) 
$$

Where 
* $\mathcal{D}$ is the memory buffer for experience replay, containing multiple episode samples $(\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}’)$
*  $f_{\theta_i}(\epsilon_i; o_i)$ is the reparameterized action using the policy network and random noise $\epsilon_i$ sampled from a standard normal distribution. 

$$f_{\theta_i}(\epsilon_i; o_i)= \mu_{\theta_i}(o_i) + \sigma_{\theta_i}(o_i) \cdot \epsilon_i, \epsilon_i \sim \mathcal{N}(0, 1)$$

$$\nabla_{\theta_i} a_i = \nabla_{\theta_i} f_{\theta_i}(\epsilon_i; o_i)$$

#### Automatic Entropy Tuning

SAC is brittle with respect to the temperature parameter. Unfortunately it is difficult to adjust temperature, because the entropy can vary unpredictably both across tasks and during training as the policy becomes better. An improvement on SAC formulates a constrained optimization problem, while maximizing the expected return, the policy should satisfy a minimum entropy constraint. The optimization problem is:
$$ J(\alpha_i) = \mathbb{E}_{a_i \sim \pi_i, o_i \sim \mathcal{D}} \left[ -\alpha_i \log \pi_i(a_i|o_i) - \alpha_i \mathcal{H}_{\text{target}} \right] $$
$$ \nabla_{\alpha_i} J(\alpha_i) = \mathbb{E}_{a_i \sim \pi_i, o_i \sim \mathcal{D}} \left[ - \log \pi_i(a_i|o_i) - \mathcal{H}_{\text{target}} \right] $$

where $\mathcal{H}_{\text{target}}$ is a target entropy value, typically set to $-\dim(\mathcal{A}_i)$.

#### Advantages over MADDPG and MATD3
1. **Improved Exploration**: Entropy maximization encourages exploration in uncertain regions of the state space
2. **Better Performance in Multi-modal Tasks**: Stochastic policies can represent multiple good strategies
3. **Reduced Sensitivity to Hyperparameters**: Automatic entropy tuning adapts exploration to each environment
4. **Robustness to Non-stationarity**: Entropy-regularized policies are more robust to changes in other agents' behaviors
5. **Sample Efficiency**: Typically requires fewer environment interactions than MADDPG to reach comparable performance

MASAC combines the strengths of maximum entropy reinforcement learning with multi-agent coordination, making it particularly effective for complex collaborative tasks requiring both exploration and precise coordination.

### MPE Environment

Great! 🎉 🎉 🎉  We done with theorectical part for MASAC! 🥳 

Now let's implement MASAC on MPE environment. MPE is a simple multi-agent environment where agents must learn to coordinate to solve tasks.

In [1]:
from pettingzoo.mpe import simple_spread_v3
import numpy as np

env = simple_spread_v3.parallel_env(continuous_actions=True)
# Reset the environment
observations, infos = env.reset()
# Print basic environment information
print(f"Environment: {env.metadata['name']}")
print(f"Number of agents: {len(env.agents)}")
for agent in env.agents:
    action_space = env.action_space(agent)
    obs_space = env.observation_space(agent)
    print(f"act - {agent}: {action_space} (shape: {action_space.shape}, bounds: [{action_space.low}, {action_space.high}])")
    print(f"obs - {agent}: {obs_space} (shape: {obs_space.shape})")

Environment: simple_spread_v3
Number of agents: 3
act - agent_0: Box(0.0, 1.0, (5,), float32) (shape: (5,), bounds: [[0. 0. 0. 0. 0.], [1. 1. 1. 1. 1.]])
obs - agent_0: Box(-inf, inf, (18,), float32) (shape: (18,))
act - agent_1: Box(0.0, 1.0, (5,), float32) (shape: (5,), bounds: [[0. 0. 0. 0. 0.], [1. 1. 1. 1. 1.]])
obs - agent_1: Box(-inf, inf, (18,), float32) (shape: (18,))
act - agent_2: Box(0.0, 1.0, (5,), float32) (shape: (5,), bounds: [[0. 0. 0. 0. 0.], [1. 1. 1. 1. 1.]])
obs - agent_2: Box(-inf, inf, (18,), float32) (shape: (18,))


### Model Architecture

MASAC uses a simple neural network architecture for the actor and critic networks. The actor network outputs the mean and standard deviation of a Gaussian distribution over actions, while the critic network takes joint observations and actions as input.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def init_weights(module, init_w=3e-3):
    if isinstance(module, nn.Linear):
            # Use Kaiming initialization for ReLU layers
            nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(module.bias)

LOG_STD_MAX = 2
LOG_STD_MIN = -20

class Actor(nn.Module):
    """Actor (Policy) Model."""
    def __init__(self, state_size, action_size, hidden_sizes=(64, 64)):
        """
        Initialize parameters and build model.
        
        Args:
            state_size (int): Dimension of each state
            action_size (int): Dimension of each action
            hidden_sizes (tuple): Sizes of hidden layers
        """
        super(Actor, self).__init__()

        self._actor = nn.Sequential(
            nn.Linear(state_size, hidden_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], action_size * 2),
        )

        self.apply(init_weights)

        # For final layer with tanh, use a small initialization:
        nn.init.xavier_uniform_(self._actor[-1].weight, gain=1)
        nn.init.zeros_(self._actor[-1].bias)
    
    def forward(self, state):
        """Build an actor (policy) network that maps states -> actions."""
        x = self._actor(state)
        mean, log_std = x.chunk(2, dim=-1)
        return mean, log_std
    
    def sample(self, state, compute_log_prob=True, deterministic=False):
        """Sample an action from the policy."""
        mean, log_std = self.forward(state)

        if deterministic:
             return torch.tanh(mean), None
        
        log_std = torch.clamp(log_std, LOG_STD_MIN, LOG_STD_MAX) # [1e-8, 1e2]
                 
        base_distribution = torch.distributions.Normal(mean, log_std.exp())
        # additional transforms to run on top
        # Make tanh to bound between [-1, 1]
        tanh_transform = torch.distributions.transforms.TanhTransform(cache_size=1)
        squashed_dist = torch.distributions.TransformedDistribution(base_distribution, [tanh_transform,])

        # Get squashed action
        action = squashed_dist.rsample()

        if not compute_log_prob:
            return action, None
        
        log_prob = squashed_dist.log_prob(action).sum(-1, keepdim=True)
        # this equal to 
        # log_prob = base_distribution.log_prob(action)
        # log_prob -=  2*(np.log(2) - action - F.softplus(-2 * action))
        # log_prob = log_prob.sum(-1, keepdim=True)

        return action, log_prob


class Critic(nn.Module):
    """Critic (Value) Model"""
    def __init__(self, state_size, action_size, hidden_sizes=(64, 64)):
        """
        Initialize parameters and build model.
        
        Args:
            state_size (int): Dimension of each state
            action_size (int): Dimension of each action
            hidden_sizes (tuple): Sizes of hidden layers
        """
        super(Critic, self).__init__()

        self._critic1 = nn.Sequential(
            nn.Linear(state_size + action_size, hidden_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], 1)
        )

        self._critic2 = nn.Sequential(
            nn.Linear(state_size + action_size, hidden_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], 1)
        )

        self.apply(init_weights)
    
    def forward(self, state, action):
        """Build a critic (value) network that maps (state, action) pairs -> 2 Q-values."""
        x = torch.cat([state, action], dim=-1)
        q1 = self._critic1(x)
        q2 = self._critic2(x)
        return q1, q2


### Replay Buffer

MASAC uses a shared replay buffer to store experiences from all agents similar to MADDPG and MATD3. Each experience tuple contains the observations, actions, rewards, and next observations for all agents in the environment.

In [3]:
import numpy as np
import torch
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ReplayBuffer:
    """Replay Buffer for multi agent environments with per-agent storage in separate arrays."""

    def __init__(self, buffer_size, batch_size, agents, state_sizes, action_sizes):
        """
        Initialize the ReplayBuffer with per-agent storage.
        
        Args:
            buffer_size (int): Maximum size of the buffer
            batch_size (int): Size of each training batch
            agents (list): List of agent IDs (e.g., from env.agents)
            state_sizes (list): List of state sizes per agent (e.g., [14, 10, 10])
            action_sizes (list): List of action sizes per agent (e.g., [7, 4, 5])
        """
        self.buffer_size = buffer_size
        self.batch_size = batch_size
        self.agents = agents
        self.num_agents = len(agents)
        self.state_sizes = state_sizes
        self.action_sizes = action_sizes

        # Initialize per-agent buffers as separate arrays
        self.states_buffer = []
        self.actions_buffer = []
        self.rewards_buffer = []
        self.next_states_buffer = []
        self.dones_buffer = []

        # Create separate buffers for each agent
        for i in range(self.num_agents):
            self.states_buffer.append(np.zeros((buffer_size, state_sizes[i]), dtype=np.float32))
            self.actions_buffer.append(np.zeros((buffer_size, action_sizes[i]), dtype=np.float32))
            self.rewards_buffer.append(np.zeros(buffer_size, dtype=np.float32))
            self.next_states_buffer.append(np.zeros((buffer_size, state_sizes[i]), dtype=np.float32))
            self.dones_buffer.append(np.zeros(buffer_size, dtype=np.uint8))
        
        self.position = 0
        self.size = 0

    def add(self, states, actions, rewards, next_states, dones):
        """
        Add a new experience to the buffer.
        
        Args:
            states (list): List of states per agent (variable sizes)
            actions (list): List of actions per agent (variable sizes)
            rewards (list or np.ndarray): Rewards per agent [num_agents]
            next_states (list): List of next states per agent (variable sizes)
            dones (list or np.ndarray): Done flags per agent [num_agents]
        """
        # Store experience for each agent
        for i in range(self.num_agents):
            self.states_buffer[i][self.position] = states[i]
            self.actions_buffer[i][self.position] = actions[i]
            # Store reward and done as scalars
            self.rewards_buffer[i][self.position] = rewards[i]
            self.next_states_buffer[i][self.position] = next_states[i]
            self.dones_buffer[i][self.position] = dones[i]
        
        # Update position and size
        self.position = (self.position + 1) % self.buffer_size
        self.size = min(self.size + 1, self.buffer_size)
    
    def sample(self):
        """
        Sample a batch of experiences from the buffer.
        
        Returns:
            tuple: (states_batch, actions_batch, rewards_batch, next_states_batch, dones_batch, 
                   states_full, next_states_full, actions_full)
                  Each is a tensor with appropriate shape for MADDPG training
        """
        # Sample indices
        indices = np.random.choice(self.size, self.batch_size, replace=False)
        
        # Initialize tensors for each agent
        states_batch = []
        actions_batch = []
        rewards_batch = []
        next_states_batch = []
        dones_batch = []
        
        # Collect experiences for each agent
        for i in range(self.num_agents):
            # Get data for this agent
            agent_states = self.states_buffer[i][indices]  # [batch_size, state_size_i]  
            agent_actions = self.actions_buffer[i][indices]  # [batch_size, action_size_i]
            agent_rewards = self.rewards_buffer[i][indices]  # [batch_size]
            agent_next_states = self.next_states_buffer[i][indices]  # [batch_size, state_size_i]
            agent_dones = self.dones_buffer[i][indices]  # [batch_size]
            
            # Convert to tensors
            states_batch.append(torch.tensor(agent_states).float().to(device))  # [agent_idx, batch_size, state_size_i]
            actions_batch.append(torch.tensor(agent_actions).float().to(device))  # [agent_idx, batch_size, action_size_i]
            rewards_batch.append(torch.tensor(agent_rewards).float().to(device))  # [agent_idx, batch_size]
            next_states_batch.append(torch.tensor(agent_next_states).float().to(device))  # [agent_idx, batch_size, state_size_i]
            dones_batch.append(torch.tensor(agent_dones).float().to(device))  # [agent_idx, batch_size]
        
        # Stack rewards and dones directly without squeeze
        rewards_batch = torch.stack(rewards_batch).unsqueeze(-1)  # [num_agents, batch_size, 1]
        dones_batch = torch.stack(dones_batch).unsqueeze(-1)  # [num_agents, batch_size, 1]
        
        # Create full state and action tensors for centralized critic
        states_full = torch.cat(states_batch, dim=-1)  # [batch_size, sum(state_sizes)]
        next_states_full = torch.cat(next_states_batch, dim=-1)  # [batch_size, sum(state_sizes)]
        actions_full = torch.cat(actions_batch, dim=-1)  # [batch_size, sum(action_sizes)]
        
        return (states_batch, actions_batch, rewards_batch, next_states_batch, 
                dones_batch, states_full, next_states_full, actions_full)

    def __len__(self):
        """Return the current size of the buffer."""
        return self.size

### SAC Agent - (multi-agent)

Define individual SAC agents for each agent in the environment. Each agent maintains its own actor and centralized critic networks, target networks, and optimizers.

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SACAgent:
    """
    SAC Agent with Actor and Centralized Critic Networks.
    """
    def __init__(self, state_size, action_size, total_state_size, total_action_size, hidden_sizes=(64, 64), 
                 actor_lr=1e-4, critic_lr=1e-3, tau=1e-3, alpha=0.2, autotune_alpha=True):
        """
        Initialize a SAC agent.
        
        Args:
            state_size (int): Dimension of the state space
            action_size (int): Dimension of the action space
            total_state_size (int): Total dimension of all agents' states (for centralized critic)
            total_action_size (int): Total dimension of all agents' actions (for centralized critic)
            hidden_sizes (tuple): Sizes of hidden layers for networks
            actor_lr (float): Learning rate for the actor
            critic_lr (float): Learning rate for the critic (not used here, kept for compatibility)
            tau (float): Soft update parameter
        """
        self.state_size = state_size
        self.action_size = action_size
        self.tau = tau
        
        # Actor Network
        self.actor = Actor(state_size, action_size, hidden_sizes).to(device)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr)

        # Critic Networks (Local and Target) - Centralized Critic
        self.critic = Critic(total_state_size, total_action_size, hidden_sizes).to(device)
        self.critic_target = Critic(total_state_size, total_action_size, hidden_sizes).to(device)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=critic_lr)

        # Alpha parameter for automatic entropy tuning
        if autotune_alpha:
            # Target entropy is -|A|, where |A| is the action space dimensionality (SAC paper) (e.g. -float(action_size))
            self.target_entropy = -torch.prod(torch.tensor(action_size, device=device)).item()

            # Initialize log_alpha as a learnable parameter, ensuring alpha = exp(log_alpha) >= 0
            self.log_alpha = torch.tensor(torch.log(alpha), requires_grad=True, device=device)
            # Optimize log_alpha with Adam
            self.alpha_optimizer = optim.Adam([self.log_alpha], lr=actor_lr)
        else:
            # Fixed alpha if autotune is disabled
            self.log_alpha = torch.tensor(torch.log(alpha), device=device, requires_grad=False)


        self.hard_update(self.critic_target, self.critic)
    
    @property
    def alpha(self):
        """Alpha is computed as exponential of log_alpha."""
        return self.log_alpha.exp()

    def act(self, state):
        """
        Returns actions for given state as per current policy.
        
        Args:
            state: Current state
            add_noise (bool): Whether to add noise for exploration
            noise_scale (float): Gaussian noise scale
        """
        state = torch.from_numpy(state).float().to(device)
        
        self.actor.eval()
        with torch.no_grad():
            # Get action from network (already scaled to [action_low, action_high])
            action = self.actor(state).cpu().data.numpy()
        self.actor.train()

        # Return action in range [-1.0, 1.0] - tanh squashed
        return action

    def act_target(self, state):
        """
        Returns actions for given state as per current target policy.
        Keeps gradients for learning.
        Args:
            state: Current state (tensor)
        Returns:
            action: Action from target policy (tensor)
        """
        # Assume state is already a tensor
        if not isinstance(state, torch.Tensor):
            state = torch.from_numpy(state).float().to(device)
            
        # Return tensor directly (with gradients) in range [-1, 1]
        return self.actor_target(state)
    
    def hard_update(self, target, source):
        """Hard update model parameters.
        θ_target = θ_source
        """
        for target_param, param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(param.data)

### MASAC Class - Orchestrating Multi-Agent Training

The MASAC class orchestrates the training process for all agents in the environment. It initializes the agents, replay buffer, and handles the training loop for the agents. The training loop consists of collecting experiences from the environment, updating the replay buffer, and training the agents using the shared replay buffer.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import os

from pettingzoo import mpe

from helpers.utils import Logger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MASAC:
    """
    Multi-Agent Soft Actor-Critic (MASAC) with centralized critics and decentralized actors.
    """
    def __init__(self,
                env_name: str = "simple_spread_v3",
                buffer_size=int(1e6),
                batch_size=512,
                hidden_sizes=(64, 64),
                actor_lr=3e-4, critic_lr=3e-4, gamma=0.95, tau=1e-2,
                alpha=0.2, autotune_alpha=True,
                warmup_steps=2000,
                update_every=25,
                max_steps=25,
                device: torch.device = torch.device("cpu"), use_logger = True):
        """
        Initialize the MASAC Agent.

        Args:
            env_name (str): PettingZoo MPE environment name
            buffer_size (int): Maximum size of the replay buffer
            batch_size (int): Size of each training batch
            hidden_sizes (tuple): Sizes of hidden layers for networks
            actor_lr (float): Learning rate for the actor
            critic_lr (float): Learning rate for the critic
            gamma (float): Discount factor
            tau (float): Soft update parameter
            alpha (float): Entropy regularization weight
            autotune_alpha (bool): Whether to autotune alpha
            warmup_steps (int): Number of warmup steps before learning starts
            use_noise_decay (bool): Whether to decay noise
            update_every (int): Update networks every n steps
            max_steps (int): Maximum number of steps per episode
            device (torch.device): Device to run the networks on (default: cpu)
        """
        try:
            env_func = getattr(mpe, env_name)  # Get the environment function (e.g., simple_spread_v3)
            self.env = env_func.parallel_env(max_cycles=max_steps, continuous_actions=True)
            self.eval_env = env_func.parallel_env(max_cycles=max_steps, continuous_actions=True)
        except AttributeError:
            raise ValueError(f"Environment '{env_name}' not found in pettingzoo.mpe. "
                            "Check the name (e.g., 'simple_spread_v3', 'simple_tag_v3').")

        self.device = device

        # Get environment information
        self.env.reset() # to get agents information
        self.agent_ids = self.env.agents
        self.num_agents = len(self.agent_ids)
        self.state_sizes = [self.env.observation_space(agent).shape[0] for agent in self.agent_ids]
        self.action_sizes = [self.env.action_space(agent).shape[0] for agent in self.agent_ids]
        # We trust all agents have the same action space
        self.action_low = self.env.action_space(self.agent_ids[0]).low[0].item()
        self.action_high = self.env.action_space(self.agent_ids[0]).high[0].item()
        self.action_range = self.action_high - self.action_low
        self.env.close() # close the environment

        # hyperparameters
        self.gamma = gamma
        self.tau = tau
        self.update_every = update_every
        self.alpha = alpha
        self.autotune_alpha = autotune_alpha
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        
        # Calculate total state and action sizes for centralized critic
        self.total_state_size = sum(self.state_sizes)
        self.total_action_size = sum(self.action_sizes)
        
        # Create agents
        self.sac_agents = []
        for i in range(self.num_agents):
            agent = SACAgent(
                self.state_sizes[i], 
                self.action_sizes[i],
                self.total_state_size,
                self.total_action_size, 
                hidden_sizes=hidden_sizes,
                actor_lr=actor_lr, 
                critic_lr=critic_lr, 
                tau=tau,
                alpha=alpha,
                autotune_alpha=autotune_alpha,
                action_low=self.action_low,
                action_high=self.action_high
            )
            self.sac_agents.append(agent)
        
        # Replay memory
        self.replay_memory = ReplayBuffer(buffer_size, 
                                          batch_size, 
                                          self.agent_ids, 
                                          self.state_sizes, 
                                          self.action_sizes)
        
        if use_logger:

            # Initialize logger
            run_name = (
                f"alr{actor_lr}_clr{critic_lr}_bs{batch_size}_"
                f"hs{hidden_sizes}_g{self.gamma}_t{tau}_"
                f"al{alpha}_aut{autotune_alpha}_"
                f"ue{update_every}_ms{max_steps}"
            )
            run_name = "".join(run_name)
            self.logger = Logger(run_name=run_name, env=self.env.metadata['name'], algo="MASAC")
            # Log hyperparameters
            self.logger.log_hyperparameters({
                "env_name": env_name,
                "buffer_size": buffer_size,
                "batch_size": batch_size,
                "hidden_sizes": hidden_sizes,
                "max_steps": max_steps,
                "actor_lr": actor_lr,
                "critic_lr": critic_lr,
                "gamma": self.gamma,
                "tau": self.tau,
                "alpha": alpha,
                "autotune_alpha": autotune_alpha,
                "warmup_steps": warmup_steps,
                "update_every": update_every
            })

    def act(self, states):
        """
        Get actions from all agents based on current policy.
        Args:
            states (list): List of states for each agent
        """
        actions = [agent.act(state) for agent, state in zip(self.sac_agents, states)]
        return actions
    
    def scale_action(self, action):
        """Scale action from [-1, 1] to [action_low, action_high]."""
        return self.action_low + 0.5 * (action + 1.0) * self.action_range

    def unscale_action(self, action):
        """Unscale action from [action_low, action_high] to [-1, 1]."""
        return 2.0 * (action - self.action_low) / self.action_range - 1.0
    
    def _rollout_step(self, observations, global_step=0):
        """
        Take a step in the environment using the current policy and store the experience in the replay buffer.
        Note: Called it rollout_step, because collect_rollout is widely used in RL literature.
        Args:
            observations (dict): Dictionary of observations for each agent
            global_step (int): Global step number
        Returns:
            next_observations (dict): Dictionary of next observations for each agent
            rewards (dict): Dictionary of rewards for each agent
            episode_done (bool): Whether the episode is done
        """
        # Get states for all agents
        states_list = [np.array(observations[agent], dtype=np.float32) for agent in self.agent_ids]
        
        # Get actions for all agents
        if global_step < self.warmup_steps:
            # Sample random actions during warmup period [0, 1]
            actions = {agent: self.env.action_space(agent).sample()  for agent in self.agent_ids}
            actions_list = [self.unscale_action(actions[agent]) for agent in self.agent_ids] # [-1, 1]
        else:
            actions_list = self.act(states_list) # [-1, 1]
            # Scale actions to [0, 1]
            scaled_actions_list = [self.scale_action(action) for action in actions_list]
            actions = {agent: action for agent, action in zip(self.agent_ids, scaled_actions_list)}
        
        # Take a step in the environment
        next_observations, rewards, terminations, truncations, _ = self.env.step(actions)
        
        # Check if episode is done
        dones = [terminations[agent] or truncations[agent] for agent in self.agent_ids]
        episode_done = any(dones)
        
        # Prepare data to store in replay buffer (we care only terminations)
        rewards_array = np.array([rewards[agent] for agent in self.agent_ids], dtype=np.float32)
        next_states_list = [np.array(next_observations[agent], dtype=np.float32) for agent in self.agent_ids]
        terminations_array = np.array([terminations[agent] for agent in self.agent_ids], dtype=np.uint8)
        
        # Store experience in replay buffer
        self.replay_memory.add(
            states=states_list,
            actions=actions_list,
            rewards=rewards_array,
            next_states=next_states_list,
            dones=terminations_array
        )

        return next_observations, rewards, episode_done


In [ ]:
# TODO: 
# 1. Define Env (DONE)
# 2. Define Model (DONE)
# 3. Define Buffer (DONE)
# 4. Define SAC Agent (DONE)
# 5. Define MASAC Agent 